# Part I · Block 3 — Comparing Conditions and Running a Cohort
**Tasks 6 & 7** | *Worksheet §Part I, Block 3 (35 min)*

First you will manually compare two sites (WT vs. PIK3CA E545K) using the
single-block script. Then you will run the full cohort comparison script
and interpret the mutation-level table.

In [ ]:
import sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
DATA_PATH    = PROJECT_ROOT / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH    = PROJECT_ROOT / '01-readme-experiment-description_2022-04-05.csv'
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'

SIGNAL_COL = 'ERKKTR_ratio'
print('Setup complete.')

---
## Metadata overview

A quick look at which site belongs to which mutation.

In [ ]:
meta = pd.read_csv(META_PATH, encoding='utf-8-sig')
meta.rename(columns={'Site': 'Image_Metadata_Site'}, inplace=True)
print(meta[['Image_Metadata_Site', 'Mutation']].to_string(index=False))

---
## Task 6 — Manual pairwise comparison: WT (site 1) vs. PIK3CA E545K (site 9)

Run the single-block script for both sites with fixed parameters
$r = 60$, $W = 3$, $q = 0.90$.

In [ ]:
def run_single_block(exp, site, signal, radius=60, window=3, quantile=0.9, outdir_tag=None):
    outdir = OUTPUT_ROOT / (outdir_tag or f'exp_{exp}_site_{site}_{signal}')
    # Reuse existing output if already computed with default tag
    summary_path = OUTPUT_ROOT / f'exp_{exp}_site_{site}_{signal}' / 'summary.json'
    if summary_path.exists() and outdir_tag is None:
        with open(summary_path) as f:
            return json.load(f)
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'spatiotemporal_signal_propagation.py'),
        '--data-path', str(DATA_PATH), '--meta-path', str(META_PATH),
        '--exp-id', str(exp), '--site-id', str(site),
        '--signal-col', signal,
        '--spatial-radius', str(radius),
        '--future-window-frames', str(window),
        '--jump-quantile', str(quantile),
        '--output-dir', str(OUTPUT_ROOT),
    ]
    subprocess.run(cmd, capture_output=True)
    with open(summary_path) as f:
        return json.load(f)

print('Running site 1 (WT) …', end=' ', flush=True)
s1 = run_single_block(1, 1, SIGNAL_COL)
print('done')

print('Running site 9 (PIK3CA E545K) …', end=' ', flush=True)
s9 = run_single_block(1, 9, SIGNAL_COL)
print('done')

In [ ]:
compare_cols = ['mutation', 'site_id', 'jump_threshold',
                'future_jump_rate_if_neighbor_jumps_now',
                'future_jump_rate_if_no_neighbor_jumps_now',
                'risk_difference', 'relative_risk']

df_pair = pd.DataFrame([s1, s9])[compare_cols].rename(columns={
    'jump_threshold': 'θ',
    'future_jump_rate_if_neighbor_jumps_now': 'p_exp',
    'future_jump_rate_if_no_neighbor_jumps_now': 'p_unexp',
    'risk_difference': 'RD',
    'relative_risk': 'RR',
})
df_pair

### Task 6 — Questions

In [ ]:
# (a) Is RR higher or lower for PIK3CA E545K compared to WT?
print(f"RR WT           = {s1['relative_risk']:.4f}")
print(f"RR PIK3CA E545K = {s9['relative_risk']:.4f}")
diff = s9['relative_risk'] - s1['relative_risk']
print(f"Δ RR (mutant − WT) = {diff:+.4f}")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Does hyperactive PI3K increase or decrease ERK coordination among neighbours?


In [ ]:
# (b) Can you directly compare the two θ values? Why or why not?
print(f"θ site 1 (WT)           = {s1['jump_threshold']:.4f}")
print(f"θ site 9 (PIK3CA E545K) = {s9['jump_threshold']:.4f}")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Hint: think about what Q_0.90(D+) means for each block separately.
# This anticipates Limitation L2 in Part II.


---
## Task 7 — Cohort-level analysis

Run `compare_spatiotemporal_behavior.py` to aggregate all available
blocks and compare by mutation — first for ERK, then for FoxO3A.

In [ ]:
print('Running cohort comparison for ERK …')
cmd_erk = [
    sys.executable,
    str(SCRIPTS_DIR / 'compare_spatiotemporal_behavior.py'),
    '--data-path', str(DATA_PATH),
    '--meta-path', str(META_PATH),
    '--signal-col', 'ERKKTR_ratio',
    '--group-by', 'mutation',
    '--spatial-radius', '60',
    '--future-window-frames', '3',
    '--jump-quantile', '0.9',
    '--exclude-mutations', 'ErbB2',
    '--output-dir', str(OUTPUT_ROOT),
]
r_erk = subprocess.run(cmd_erk, capture_output=True, text=True)
print(r_erk.stdout[-800:])  # tail of stdout

In [ ]:
group_erk = pd.read_csv(
    OUTPUT_ROOT / 'comparison_mutation_ERKKTR_ratio' / 'group_level_summary.csv'
)
group_erk.sort_values('mean_relative_risk', ascending=False)

In [ ]:
print('Running cohort comparison for FoxO3A …')
cmd_foxo = [
    sys.executable,
    str(SCRIPTS_DIR / 'compare_spatiotemporal_behavior.py'),
    '--data-path', str(DATA_PATH),
    '--meta-path', str(META_PATH),
    '--signal-col', 'FoxO3A_ratio',
    '--group-by', 'mutation',
    '--spatial-radius', '60',
    '--future-window-frames', '3',
    '--jump-quantile', '0.9',
    '--exclude-mutations', 'ErbB2',
    '--output-dir', str(OUTPUT_ROOT),
]
subprocess.run(cmd_foxo, capture_output=True)

group_foxo = pd.read_csv(
    OUTPUT_ROOT / 'comparison_mutation_FoxO3A_ratio' / 'group_level_summary.csv'
)
group_foxo.sort_values('mean_relative_risk', ascending=False)

In [ ]:
# ── Side-by-side comparison of mean RR across mutations ──────────────────────
rr_erk  = group_erk.set_index('comparison_group')['mean_relative_risk'].rename('RR_ERK')
rr_foxo = group_foxo.set_index('comparison_group')['mean_relative_risk'].rename('RR_FoxO3A')
combined = pd.concat([rr_erk, rr_foxo], axis=1).sort_values('RR_ERK', ascending=False)
print(combined.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(combined))
ax.bar([i - 0.2 for i in x], combined['RR_ERK'],   width=0.4, label='ERK',    color='steelblue')
ax.bar([i + 0.2 for i in x], combined['RR_FoxO3A'], width=0.4, label='FoxO3A', color='darkorange')
ax.axhline(1, color='grey', ls='--', lw=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(combined.index, rotation=20, ha='right')
ax.set_ylabel('Mean relative risk $\\overline{\\mathrm{RR}}$')
ax.set_title('Spatiotemporal coordination by mutation and signal')
ax.legend()
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# (a) Which mutation has highest/lowest mean RR for ERK?
# (b) Is ERK or FoxO3A more spatially coordinated overall?
# (c) Does the ordering fit the PI3K pathway hierarchy (PI3K → AKT ← PTEN)?


### Task 7(b) — Stability: how many blocks per group?

$\overline{\mathrm{RR}}(g)$ is fragile when `n_blocks` is small. Check this:

In [ ]:
block_level = pd.read_csv(
    OUTPUT_ROOT / 'comparison_mutation_ERKKTR_ratio' / 'block_level_summary.csv'
)
print(block_level.groupby('mutation')[['relative_risk']].agg(['mean', 'std', 'count']).round(3))

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# For groups with n_blocks = 1, what uncertainty does the mean RR carry?
